In [16]:
import numpy as np
import random

In [17]:
train_data = {
  'good': True,
  'bad': False,
  'happy': True,
  'sad': False,
  'not good': False,
  'not bad': True,
  'not happy': False,
  'not sad': True,
  'very good': True,
  'very bad': False,
  'very happy': True,
  'very sad': False,
  'i am happy': True,
  'this is good': True,
  'i am bad': False,
  'this is bad': False,
  'i am sad': False,
  'this is sad': False,
  'i am not happy': False,
  'this is not good': False,
  'i am not bad': True,
  'this is not sad': True,
  'i am very happy': True,
  'this is very good': True,
  'i am very bad': False,
  'this is very sad': False,
  'this is very happy': True,
  'i am good not bad': True,
  'this is good not bad': True,
  'i am bad not good': False,
  'i am good and happy': True,
  'this is not good and not happy': False,
  'i am not at all good': False,
  'i am not at all bad': True,
  'i am not at all happy': False,
  'this is not at all sad': True,
  'this is not at all happy': False,
  'i am good right now': True,
  'i am bad right now': False,
  'this is bad right now': False,
  'i am sad right now': False,
  'i was good earlier': True,
  'i was happy earlier': True,
  'i was bad earlier': False,
  'i was sad earlier': False,
  'i am very bad right now': False,
  'this is very good right now': True,
  'this is very sad right now': False,
  'this was bad earlier': False,
  'this was very good earlier': True,
  'this was very bad earlier': False,
  'this was very happy earlier': True,
  'this was very sad earlier': False,
  'i was good and not bad earlier': True,
  'i was not good and not happy earlier': False,
  'i am not at all bad or sad right now': True,
  'i am not at all good or happy right now': False,
  'this was not happy and not good earlier': False,
}

test_data = {
  'this is happy': True,
  'i am good': True,
  'this is not happy': False,
  'i am not good': False,
  'this is not bad': True,
  'i am not sad': True,
  'i am very good': True,
  'this is very bad': False,
  'i am very sad': False,
  'this is bad not good': False,
  'this is good and happy': True,
  'i am not good and not happy': False,
  'i am not at all sad': True,
  'this is not at all good': False,
  'this is not at all bad': True,
  'this is good right now': True,
  'this is sad right now': False,
  'this is very bad right now': False,
  'this was good earlier': True,
  'i was not happy and not good earlier': False,
}

In [18]:
vocab = list(set([w for text in train_data.keys() for w in text.split(' ')]))
vocab_size = len(vocab)

In [19]:
print("%d unique words found in the train set" %vocab_size)

18 unique words found in the train set


In [20]:
word_to_index = {w : i for i, w in enumerate(vocab)}
index_to_word = {i : w for i, w in enumerate(vocab)}

In [21]:
def create_inputs(text):
    inputs = []
    for w in text.split(' '):
        v = np.zeros((vocab_size, 1))
        v[word_to_index[w]] = 1
        inputs.append(v)
    return inputs

In [22]:
class Rnn:

    def __init__(self, input_size, output_size, hidden_size = 64):
        self.wxh = np.random.randn(hidden_size, input_size) / 1000
        self.whh = np.random.randn(hidden_size, hidden_size) / 1000
        self.why = np.random.randn(output_size, hidden_size) / 1000
        self.bh = np.zeros((hidden_size, 1))
        self.by = np.zeros((output_size, 1))

    def forward(self, inputs):
        h = np.zeros((self.whh.shape[0], 1))
        self.last_inputs = inputs
        self.last_hs = {0:h}
        for i, x in enumerate(inputs):
            h = np.tanh(self.wxh @ x + self.whh @ h + self.bh)
            self.last_hs[i+1] = h
        y = self.why @ h + self.by
        return y, h

    def backward(self,grad_out, learn_rate= 2e-2):
        n = len(self.last_inputs)
        d_l_d_why = grad_out @ self.last_hs[n].T
        d_l_d_by = grad_out
        d_l_d_wxh = np.zeros(self.wxh.shape)
        d_l_d_whh = np.zeros(self.whh.shape)
        d_l_d_bh = np.zeros(self.bh.shape)
        d_l_d_h = self.why.T @ grad_out
        for t in reversed(range(n)):
            temp = ((1 - self.last_hs[t+1]**2)*d_l_d_h)
            d_l_d_bh += temp
            d_l_d_wxh += temp * self.last_inputs[t].T
            d_l_d_whh += temp * self.last_hs[t].T
            d_l_d_h = self.whh @ temp
        for d in [d_l_d_wxh, d_l_d_whh, d_l_d_why, d_l_d_bh, d_l_d_by]:
            np.clip(d, -1, 1, out=d)
        self.wxh -= learn_rate * d_l_d_wxh
        self.whh -= learn_rate * d_l_d_whh
        self.why -= learn_rate * d_l_d_why
        self.bh -= learn_rate * d_l_d_bh
        self.by -= learn_rate * d_l_d_by

In [23]:
def softmax(input):
    return np.exp(input)/ np.sum(np.exp(input))

In [24]:
rnn = Rnn(vocab_size, 2)

for x,y in train_data.items():
    inputs = create_inputs(x)
    true_out = int(y)
    out, h = rnn.forward(inputs)
    probs= softmax(out)
    d_l_d_y = probs
    d_l_d_y[true_out] -= 1
    rnn.backward(d_l_d_y)
    print(probs)

[[ 0.5000011]
 [-0.5000011]]
[[-0.50500202]
 [ 0.50500202]]
[[ 0.50004804]
 [-0.50004804]]
[[-0.50495434]
 [ 0.50495434]]
[[-0.49990021]
 [ 0.49990021]]
[[ 0.50509675]
 [-0.50509675]]
[[-0.4999553]
 [ 0.4999553]]
[[ 0.5050434]
 [-0.5050434]]
[[ 0.49999737]
 [-0.49999737]]
[[-0.50500571]
 [ 0.50500571]]
[[ 0.50004438]
 [-0.50004438]]
[[-0.50495795]
 [ 0.50495795]]
[[ 0.50009301]
 [-0.50009301]]
[[ 0.49509516]
 [-0.49509516]]
[[-0.50985703]
 [ 0.50985703]]
[[-0.50475888]
 [ 0.50475888]]
[[-0.49971342]
 [ 0.49971342]]
[[-0.49471572]
 [ 0.49471572]]
[[-0.48976984]
 [ 0.48976984]]
[[-0.48487194]
 [ 0.48487194]]
[[ 0.51997032]
 [-0.51997032]]
[[ 0.51477492]
 [-0.51477492]]
[[ 0.50962981]
 [-0.50962981]]
[[ 0.50453747]
 [-0.50453747]]
[[-0.5005096]
 [ 0.5005096]]
[[-0.49550613]
 [ 0.49550613]]
[[ 0.50944787]
 [-0.50944787]]
[[ 0.50435668]
 [-0.50435668]]
[[ 0.49931254]
 [-0.49931254]]
[[-0.50567948]
 [ 0.50567948]]
[[ 0.49937366]
 [-0.49937366]]
[[-0.50562052]
 [ 0.50562052]]
[[-0.50055983]
 

In [30]:
def process_data(data, backprop=True):
    items = list(data.items())
    random.shuffle(items)
    loss = 0
    n_accurate = 0
    for x,y in items:
        inputs = create_inputs(x)
        true_out = int(y)
        out, _ = rnn.forward(inputs)
        probs = softmax(out)
        loss -= np.log(probs[true_out, 0])
        n_accurate += int(np.argmax(probs) == true_out)
        if backprop:
            d_l_d_y = probs
            d_l_d_y[true_out] -= 1
            rnn.backward(d_l_d_y)
    return loss/len(data), n_accurate/len(data)

In [28]:
for epoch in range(1000):
    train_loss, train_accuracy = process_data(train_data)
    if epoch % 100 == 99:
        print('--- Epoch %d' % (epoch + 1))
        print('Train:\tLoss %.3f | Accuracy: %.3f' % (train_loss, train_accuracy))
        test_loss, test_accuracy = process_data(test_data, backprop=False)
        print('Test:\tLoss %.3f | Accuracy: %.3f' % (test_loss, test_accuracy))

--- Epoch 100
Train:	Loss 0.661 | Accuracy: 0.655
Test:	Loss 0.723 | Accuracy: 0.400
--- Epoch 200
Train:	Loss 0.550 | Accuracy: 0.672
Test:	Loss 0.854 | Accuracy: 0.500
--- Epoch 300
Train:	Loss 0.413 | Accuracy: 0.810
Test:	Loss 0.669 | Accuracy: 0.600
--- Epoch 400
Train:	Loss 0.271 | Accuracy: 0.879
Test:	Loss 0.365 | Accuracy: 0.800
--- Epoch 500
Train:	Loss 0.016 | Accuracy: 1.000
Test:	Loss 0.026 | Accuracy: 1.000
--- Epoch 600
Train:	Loss 0.004 | Accuracy: 1.000
Test:	Loss 0.008 | Accuracy: 1.000
--- Epoch 700
Train:	Loss 0.002 | Accuracy: 1.000
Test:	Loss 0.005 | Accuracy: 1.000
--- Epoch 800
Train:	Loss 0.002 | Accuracy: 1.000
Test:	Loss 0.004 | Accuracy: 1.000
--- Epoch 900
Train:	Loss 0.001 | Accuracy: 1.000
Test:	Loss 0.003 | Accuracy: 1.000
--- Epoch 1000
Train:	Loss 0.001 | Accuracy: 1.000
Test:	Loss 0.002 | Accuracy: 1.000
